<a href="https://colab.research.google.com/github/syltaer-utp/Project-S100/blob/main/S100_Equipo2_ParteI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S100 — Parte I: Exploración preliminar
## Gasto en salud y esperanza de vida (América + Europa, 2000–2024)

**Equipo 2**  
Universidad Tecnológica de Panamá — Maestría en Analítica de Datos  
*Cuaderno que acompaña el white paper `S100_Equipo2_ParteI.pdf`*

Este cuaderno realiza la exploración preliminar exigida por la consignia del curso.

## 1. Carga de librerías y configuración

Importamos las librerías necesarias para la manipulación de datos y la visualización.  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo y tamaño por defecto de las figuras
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["figure.dpi"] = 100

## 2. Carga del conjunto de datos original

El archivo `health_panel.csv` es un panel país-año construido a partir de fuentes oficiales (World Bank, WHO y OECD).  
Contiene indicadores de esperanza de vida, gasto en salud, mortalidad, PIB y recursos sanitarios desde 1960 hasta 2024.

In [2]:
DATA_PATH = "health_panel.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape original: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Rango de años: {df['year'].min()} – {df['year'].max()}")
print(f"Países / entidades únicas: {df['country_name'].nunique()}")
print("\nColumnas disponibles:")
print(df.columns.tolist())

Shape original: 17,210 filas × 32 columnas
Rango de años: 1960 – 2024
Países / entidades únicas: 265

Columnas disponibles:
['country_code', 'country_name', 'year', 'life_expectancy_total', 'life_expectancy_female', 'life_expectancy_male', 'life_expectancy_gender_gap', 'le_total_yoy_change', 'region', 'income_group', 'iso2_code', 'health_spend_pct_gdp', 'health_spend_per_capita_usd', 'spend_pct_gdp_yoy_change', 'spend_per_capita_yoy_pct', 'hospital_beds_per_1000', 'physicians_per_1000', 'nurses_midwives_per_1000', 'total_health_workers_per_1000', 'infant_mortality_per_1000', 'under5_mortality_per_1000', 'maternal_mortality_per_100k', 'infant_mortality_yoy_change', 'under5_mortality_yoy_change', 'gdp_per_capita_usd', 'population_total', 'log_gdp_per_capita', 'gdp_per_capita_yoy_pct', 'le_per_gdp_point', 'le_per_1k_spend', 'le_spend_residual', 'efficiency_score']


In [3]:
# Valores faltantes en porcentaje que ayude a la visualización
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
print("\n% de valores faltantes en el dataset")
print(missing_pct.round(1).to_string())


% de valores faltantes en el dataset
income_group                     100.0
iso2_code                        100.0
region                           100.0
nurses_midwives_per_1000          79.4
spend_per_capita_yoy_pct          68.2
spend_pct_gdp_yoy_change          68.2
physicians_per_1000               67.9
health_spend_per_capita_usd       66.8
le_spend_residual                 66.8
efficiency_score                  66.8
le_per_1k_spend                   66.8
le_per_gdp_point                  66.8
health_spend_pct_gdp              66.8
hospital_beds_per_1000            65.9
total_health_workers_per_1000     65.9
maternal_mortality_per_100k       45.2
infant_mortality_yoy_change       22.9
under5_mortality_yoy_change       22.7
infant_mortality_per_1000         21.5
under5_mortality_per_1000         21.3
gdp_per_capita_yoy_pct            16.5
gdp_per_capita_usd                15.4
log_gdp_per_capita                15.4
le_total_yoy_change                1.7
country_code              

## 3. Creación del Subset: América + Europa y años ≥ 2000

El archivo original tiene más de 17 000 filas e incluye agregados regionales y muchos países.  
Para mantener el análisis manejable y enfocado, limitamos el estudio a:

- **Países individuales** de América y Europa (sin agregados como “World” o “Latin America & Caribbean”).
- **Años a partir de 2000**, buscamos tomar en cuenta datos cercanos a la actualidad, por lo que solo tomamos a partir de este siglo.

A continuación definimos las listas de países, aplicamos el filtro y creamos la variable `continent`.

In [4]:
americas = [
    "Antigua and Barbuda", "Argentina", "Bahamas, The", "Barbados", "Belize",
    "Bolivia", "Brazil", "Canada", "Chile", "Colombia", "Costa Rica", "Cuba",
    "Dominica", "Dominican Republic", "Ecuador", "El Salvador", "Grenada",
    "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "Paraguay", "Peru", "St. Kitts and Nevis",
    "St. Lucia", "St. Vincent and the Grenadines", "Suriname",
    "Trinidad and Tobago", "United States", "Uruguay", "Venezuela, RB"
]

europe = [
    "Albania", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Hungary", "Iceland",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia", "Norway",
    "Poland", "Portugal", "Romania", "Russian Federation", "Serbia",
    "Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland",
    "Ukraine", "United Kingdom"
]

df_ae = df[df["country_name"].isin(americas + europe)].copy()
df_ae["continent"] = np.where(df_ae["country_name"].isin(americas), "Americas", "Europe")

df_subset = df_ae[df_ae["year"] >= 2000].copy()

print(f"Shape del subset (América + Europa, ≥ 2000): {df_subset.shape[0]:,} filas × {df_subset.shape[1]} columnas")
print(f"Años: {df_subset['year'].min()} – {df_subset['year'].max()}")
print(f"Países: {df_subset['country_name'].nunique()}")

Shape del subset (América + Europa, ≥ 2000): 1,875 filas × 33 columnas
Años: 2000 – 2024
Países: 75


## 4. Análisis de valores faltantes

Revisamos el porcentaje de faltantes en las variables. Esto nos permite ver, con datos, en qué columnas hay más huecos dentro del subset elegido.

In [5]:
# Vista en Porcentaje que ayuda a la visualización
missing_pct = (df_subset.isnull().mean() * 100).sort_values(ascending=False)
print("\n% de valores faltantes en el subset")
print(missing_pct.round(1).to_string())


% de valores faltantes en el subset
region                           100.0
iso2_code                        100.0
income_group                     100.0
physicians_per_1000               32.6
nurses_midwives_per_1000          29.7
total_health_workers_per_1000     25.4
hospital_beds_per_1000            21.3
spend_pct_gdp_yoy_change           8.6
spend_per_capita_yoy_pct           8.6
health_spend_per_capita_usd        4.7
efficiency_score                   4.7
le_spend_residual                  4.7
le_per_1k_spend                    4.7
le_per_gdp_point                   4.6
health_spend_pct_gdp               4.6
maternal_mortality_per_100k        4.0
log_gdp_per_capita                 0.2
gdp_per_capita_usd                 0.2
country_code                       0.0
life_expectancy_total              0.0
country_name                       0.0
life_expectancy_gender_gap         0.0
life_expectancy_male               0.0
le_total_yoy_change                0.0
year                       

## 5. Estadísticos descriptivos (Tabla 1 del white paper)

Calculamos media, mediana, mínimo y máximo de la variable objetivo y de tres predictoras relevantes.  


In [7]:
desc_vars = [
    "life_expectancy_total",
    "health_spend_per_capita_usd",
    "gdp_per_capita_usd",
    "infant_mortality_per_1000"
]

desc = df_subset[desc_vars].describe().T[["mean", "50%", "min", "max"]]
desc = desc.rename(columns={"50%": "median"})
desc["missing_pct"] = df_subset[desc_vars].isnull().mean() * 100

print("Estadísticos descriptivos (subset completo LE + gasto):")
print(desc.round(2).to_string())

print("\n--- Esperanza de vida por continente ---")
print(
    df_subset.groupby("continent")["life_expectancy_total"]
    .describe()[["count", "mean", "50%", "min", "max"]]
    .round(2)
)

Estadísticos descriptivos (subset completo LE + gasto):
                                 mean    median     min        max  missing_pct
life_expectancy_total           75.51     75.54   45.58      84.41         0.00
health_spend_per_capita_usd   1770.71    779.39   18.61   13473.19         4.75
gdp_per_capita_usd           20386.59  11556.30  440.54  137781.68         0.21
infant_mortality_per_1000       11.00      7.70    1.50      79.30         0.00

--- Esperanza de vida por continente ---
            count   mean    50%    min    max
continent                                    
Americas    875.0  73.04  72.90  45.58  82.16
Europe     1000.0  77.67  78.36  65.03  84.41


## 6. Figura 1 — Histograma de la variable objetivo

La distribución de la esperanza de vida es el gráfico principal que debe aparecer en el white paper.  